In [29]:
import pandas as pd
import numpy as np
import plotly.express as px

df = pd.read_csv('../data/data1.csv', encoding='cp949')

df


# 1. 데이터 선택
# 2. EDA
# 3. 시각화 or KPI 설정
# 4. KPI 지수 계산
# 5. 대시보드 만들기


,연도,월,분기,청코드,내외항구분,수출입구분명,시설코드,시설명,부두구분명,아외국구분,적공구분,컨테이너수(10피트),컨테이너수(20피트),컨테이너수(40피트),컨테이너수(기타),전체개수,전체물동량
0,2024,3,1,신항,외항,수입,6,신항 W 정박지,일반부두,외국선,적컨,0,3,8,0,11,19.00
1,2024,4,2,신항,외항,수입,6,신항 W 정박지,일반부두,아국선,적컨,0,9,3,0,12,15.00
2,2024,11,4,신항,외항,수입환적,7,신항 U 정박지,일반부두,아국선,공컨,0,1,0,0,1,1.00
3,2024,11,4,신항,외항,수입,7,신항 U 정박지,일반부두,아국선,공컨,0,57,2,0,59,61.00
4,2024,11,4,신항,외항,수입환적,7,신항 U 정박지,일반부두,아국선,적컨,0,31,100,0,131,231.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2404,2024,7,3,북항,외항,수입환적,8,자성대 부두,컨테이너부두,외국선,적컨,0,8769,6434,0,15203,21637.00
2405,2024,7,3,북항,외항,수출,8,자성대 부두,컨테이너부두,외국선,적컨,0,7719,6438,2,14159,20599.50
2406,2024,7,3,북항,외항,수입,8,자성대 부두,컨테이너부두,외국선,적컨,0,7818,7253,0,15071,22324.00
2407,2024,11,4,북항,외항,수출,8,자성대 부두,컨테이너부두,외국선,적컨,0,2519,2018,3,4540,6561.75


In [36]:
df['전체물동량'].isna().sum() # 10피트가 0.5로 잡히기 때문 (ㅇㅅㅇ?)

np.int64(0)

In [26]:
print(df['연도'].unique().tolist())
print(df['월'].unique().tolist())
print(df['분기'].unique().tolist())
print(df['청코드'].unique().tolist())
print(df['내외항구분'].unique().tolist())
print(df['수출입구분명'].unique().tolist())

print(df.columns.to_list)

print(df.info())


# df[df[df['내외항구분']=='내항']==df[df['수출입구분명']=='수출환적']]



[2024]
[3, 4, 11, 8, 10, 2, 6, 7, 5, 12, 9, 1]
[1, 2, 4, 3]
['신항', '북항', '감천']
['외항']
['수입', '수입환적', '수출환적', '수출']
<bound method IndexOpsMixin.tolist of Index(['연도', '월', '분기', '청코드', '내외항구분', '수출입구분명', '시설코드', '시설명', '부두구분명',
       '아외국구분', '적공구분', '컨테이너수(10피트)', '컨테이너수(20피트)', '컨테이너수(40피트)',
       '컨테이너수(기타)', '전체개수', '전체물동량'],
      dtype='str')>
<class 'pandas.DataFrame'>
RangeIndex: 2409 entries, 0 to 2408
Data columns (total 17 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   연도           2409 non-null   int64  
 1   월            2409 non-null   int64  
 2   분기           2409 non-null   int64  
 3   청코드          2409 non-null   str    
 4   내외항구분        2409 non-null   str    
 5   수출입구분명       2409 non-null   str    
 6   시설코드         2409 non-null   int64  
 7   시설명          2409 non-null   str    
 8   부두구분명        2409 non-null   str    
 9   아외국구분        2409 non-null   str    
 10  적공구분         2409 non-null   str    
 11 

In [27]:

for col in ['청코드', '내외항구분', '수출입구분명', '부두구분명','아외국구분', '적공구분'] :
    print(col,":",df[col].unique())
    print()

청코드 : <StringArray>
['신항', '북항', '감천']
Length: 3, dtype: str

내외항구분 : <StringArray>
['외항']
Length: 1, dtype: str

수출입구분명 : <StringArray>
['수입', '수입환적', '수출환적', '수출']
Length: 4, dtype: str

부두구분명 : <StringArray>
['일반부두', '컨테이너부두']
Length: 2, dtype: str

아외국구분 : <StringArray>
['외국선', '아국선']
Length: 2, dtype: str

적공구분 : <StringArray>
['적컨', '공컨']
Length: 2, dtype: str



In [35]:
df.groupby(['청코드','월']).size().unstack(level=0)
# 뭘 볼수있는거라고...? 북항이랑 신항이랑 데이터를 비교해서 다른게 맞는거래 근데 뭔소린지 모르게씀


청코드,감천,북항,신항
월,,,
1,7,87,97
2,9,87,101
3,11,94,98
4,8,77,107
5,13,75,113
6,9,70,110
7,10,79,116
8,8,70,112
9,7,89,114


## 목표
- 환적을 얼마나 많이 하는가 (규모)
- 환적을 얼마나 효율적으로 처리하는가 (효율)
    - 머라고..?
        - 머 대비 환적이 얼마나 많은가??? 맞나..????

In [40]:
# 환적 비중 계산
df['환적여부'] = df['수출입구분명'].isin(['수입환적','수출환적'])

df
# 현재 환적여부에서 false가 52%로 아닌게 더 많지만, 환적은 매년 달라진다.



,연도,월,분기,청코드,내외항구분,수출입구분명,시설코드,시설명,부두구분명,아외국구분,적공구분,컨테이너수(10피트),컨테이너수(20피트),컨테이너수(40피트),컨테이너수(기타),전체개수,전체물동량,환적여부
0,2024,3,1,신항,외항,수입,6,신항 W 정박지,일반부두,외국선,적컨,0,3,8,0,11,19.00,False
1,2024,4,2,신항,외항,수입,6,신항 W 정박지,일반부두,아국선,적컨,0,9,3,0,12,15.00,False
2,2024,11,4,신항,외항,수입환적,7,신항 U 정박지,일반부두,아국선,공컨,0,1,0,0,1,1.00,True
3,2024,11,4,신항,외항,수입,7,신항 U 정박지,일반부두,아국선,공컨,0,57,2,0,59,61.00,False
4,2024,11,4,신항,외항,수입환적,7,신항 U 정박지,일반부두,아국선,적컨,0,31,100,0,131,231.00,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2404,2024,7,3,북항,외항,수입환적,8,자성대 부두,컨테이너부두,외국선,적컨,0,8769,6434,0,15203,21637.00,True
2405,2024,7,3,북항,외항,수출,8,자성대 부두,컨테이너부두,외국선,적컨,0,7719,6438,2,14159,20599.50,False
2406,2024,7,3,북항,외항,수입,8,자성대 부두,컨테이너부두,외국선,적컨,0,7818,7253,0,15071,22324.00,False
2407,2024,11,4,북항,외항,수출,8,자성대 부두,컨테이너부두,외국선,적컨,0,2519,2018,3,4540,6561.75,False


In [51]:
zone_total = df.groupby('청코드')['전체물동량'].sum()

zone_total # 청코드별(감천,북항,신항) 전체 컨테이너 처리량

청코드
감천        9014.0
북항     6512510.0
신항    17880496.0
Name: 전체물동량, dtype: float64

In [53]:
zone_transship = df[df['환적여부']].groupby('청코드')['전체물동량'].sum()
zone_transship

청코드
감천        6192.00
북항     2314512.75
신항    11176479.25
Name: 전체물동량, dtype: float64

In [57]:
zone_share = ((zone_transship/zone_total)*100).round(2)

zone_share # 구성이 아니라 성과를 볼 수 있는 것!
# 이를 통해 안 사실
# "북항도 환적이 많을 줄 알았는데 아니였다"
# "감천은 벌크가 많이 들어오는 항구였다"

# 환적에 대한 지표를 뭘 볼건지
# 항만크레인이 컨테이너를 한번 들어옮기는 작업을 할때 몇피트건 같은 횟수로
#컨테이너 1개당 평균 몇 teu를 실어나르는가? (???????????)
# teu_factor라는 실제 용어



청코드
감천    68.69
북항    35.54
신항    62.51
Name: 전체물동량, dtype: float64

In [60]:
# 데이터의 신뢰도를 확인해봐야 함. (수기데이터일 경우 휴먼에러 가능성)

df.columns

df[df['환적여부']].groupby('청코드')['컨테이너수(기타)'].sum()

청코드
감천        0
북항       51
신항    32877
Name: 컨테이너수(기타), dtype: int64

In [64]:
teu_estimate = df['컨테이너수(10피트)']*0.5+df['컨테이너수(20피트)']*1+df['컨테이너수(40피트)']*2

# corr 다시 공부하기
# 높은 상관계수를 가지고 있기 때문에 전체물동량을 사용해도 되겠다는 판단을 해도 됨.
teu_estimate.corr(df['전체물동량'])

np.float64(0.9999898787140992)

In [71]:
# agg 뭔지 공부하기
# 각 항별 몇개를 처리했고, 물동량이 얼마나 처리됐는지 확인 40pt 짜리를 많이 깐게 더 효율이 좋은것!
grp = df.groupby(['청코드','환적여부']).agg(물동량=('전체물동량','sum'),개수=('전체개수','sum'))

grp['TEU_FACTOR'] = (grp['물동량']/grp['개수']).round(2)

# 2에 가까우면 40피트짜리를 많이 깐거라고 볼수있음!
# 북항과 비교했을 때 신항은 대형화물을 처리할 가능성이 높음
grp



물동량       개수  TEU_FACTOR
청코드 환적여부                                   
감천  False      2822.00     1516        1.86
    True       6192.00     3210        1.93
북항  False   4197997.25  2859718        1.47
    True    2314512.75  1569611        1.47
신항  False   6704016.75  4140278        1.62
    True   11176479.25  6464959        1.73